# License Plate Detection — YOLOv11 + EasyOCR Training

Based on: **Parvaiz et al. (2025)** — *Automatic Number Plate Recognition Using YOLOv11 and EasyOCR for Indian Road Conditions*

**Environment**: Google Colab with Tesla T4 GPU

**Pipeline**:
1. Install dependencies
2. Download Roboflow dataset
3. Train YOLOv11 (30 epochs, batch 16, CIoU loss)
4. Evaluate model (mAP, precision, recall)
5. Test inference with EasyOCR
6. Download trained weights

## 1. Install Dependencies

In [1]:
!pip install ultralytics roboflow easyocr opencv-python-headless pandas matplotlib seaborn

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 41.8 MB/s  0:00:00
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
    --------------------------------------- 0.5/38.8 MB 209.6 MB/s eta 0:00:01
   --- ------------------------------------ 3.4/38.8 MB 8.3 MB/s eta 0:00:05
   ---- ----------------------------------- 4.2/38.8 MB 7.1 MB/s eta 0:00:05
   ----- ---------------------------------- 5.0/38.8 MB 5.9 MB/s eta 0:00:06
   ----- ---------------------------------- 5.5/38.8 MB 5.3 MB/s eta 0:00:07
   ------ --------------------------------- 6.3/38.8 MB 5.2 MB/s eta 0:00:07
   ------- -------------------------------- 7.6/38.8 MB 5.2 MB/s eta 0:00:06
   --------- ------------------------------ 8.9/38.8 MB 5.3 MB/s eta 0:00:06
   ---------- ----------------------------- 9.7/38.8 MB 5.1 MB/s eta 0:00:06
   ----------- ---------------------------- 11.0/38.8 MB 5.2 MB/s eta 0:00:06
   -----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
postgrest 0.13.2 requires httpx<0.26,>=0.24, but you have httpx 0.28.1 which is incompatible.
storage3 0.6.1 requires httpx<0.26,>=0.24, but you have httpx 0.28.1 which is incompatible.
supabase 2.0.2 requires httpx<0.25.0,>=0.24.0, but you have httpx 0.28.1 which is incompatible.
supafunc 0.3.3 requires httpx<0.26,>=0.24, but you have httpx 0.28.1 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Download Dataset from Roboflow

Paper ref (Section 3.2): *"The system employed the 'Indian License Plate' dataset on Roboflow with around 10,000 YOLO-formatted annotated images."*

**Steps**:
1. Go to [Roboflow Universe](https://universe.roboflow.com/)
2. Search for "Indian Number Plate Detection"
3. Get your API key from Roboflow settings
4. Replace `YOUR_API_KEY` below

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="KEY")
project = rf.workspace("ashutosh-saklani").project("indian-plate-detection")
version = project.version(1)
dataset = version.download("yolov11")

## 3. Train YOLOv11

Paper ref (Section 3.3 + 6.2):
- **Model**: YOLOv11 nano (`yolo11n.pt`)
- **Epochs**: 30
- **Batch size**: 16
- **Image size**: 640x640
- **Loss function**: CIoU (Complete Intersection over Union)
- **Augmentations**: rotation, noise, brightness, contrast, horizontal flip
- **Split**: 70% train / 20% val / 10% test

In [ ]:
from ultralytics import YOLO

# Load YOLOv11 nano model (pretrained on COCO)
model = YOLO("yolo11n.pt")

# Train on the license plate dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    batch=16,
    imgsz=640,
    name="license_plate_detector",
    patience=10,       # Early stopping if no improvement for 10 epochs
    augment=True,      # Enable augmentation (rotation, noise, brightness, flip)
    verbose=True,
    plots=True         # Generate training plots
)

print("Training complete!")
print(f"Best weights saved at: runs/detect/license_plate_detector/weights/best.pt")

## 4. Evaluate the Model

Paper ref (Section 6.2): *"Detection accuracy, expressed as mAP@0.5, was 92.4%. EasyOCR character-level accuracy was around 88.2%."*

In [ ]:
# Load the best trained weights
best_model = YOLO("runs/detect/license_plate_detector/weights/best.pt")

# Validate on the validation set
metrics = best_model.val()

print(f"\n{'='*50}")
print(f"MODEL EVALUATION RESULTS")
print(f"{'='*50}")
print(f"mAP@0.5:       {metrics.box.map50:.4f}   (paper: 0.924)")
print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")
print(f"{'='*50}")

## 5. Visualize Training Metrics

Paper ref (Fig. 7): *"Training loss and metric evolution over 30 epochs."*

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Display training results
results_path = "runs/detect/license_plate_detector/results.png"
if os.path.exists(results_path):
    img = Image.open(results_path)
    plt.figure(figsize=(15, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results — Loss and Metrics Over 30 Epochs')
    plt.show()

# Display confusion matrix
cm_path = "runs/detect/license_plate_detector/confusion_matrix.png"
if os.path.exists(cm_path):
    img = Image.open(cm_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()

## 6. Test Inference with EasyOCR

Paper ref (Algorithm 1):
1. Resize image -> YOLO detect plates
2. For each plate: grayscale -> CLAHE -> skew correction -> EasyOCR
3. Regex validate the output
4. Annotate image and log to CSV

In [ ]:
import easyocr
import cv2
import re
import pandas as pd
from datetime import datetime
from google.colab.patches import cv2_imshow

# Load the best model
model = YOLO("runs/detect/license_plate_detector/weights/best.pt")

# Initialize EasyOCR reader (English — paper default)
reader = easyocr.Reader(['en'], gpu=True)

# Indian license plate regex (paper Section 3.3)
PLATE_REGEX = re.compile(r'^[A-Z]{2}\s?\d{1,2}\s?[A-Z]{1,3}\s?\d{1,4}$')

def preprocess_plate(plate_crop):
    """Grayscale -> CLAHE -> deskew (paper Eq. 3)"""
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return enhanced

# Get test images
import glob
test_images = glob.glob(f"{dataset.location}/test/images/*")[:5]  # First 5 test images

csv_rows = []

for img_path in test_images:
    img = cv2.imread(img_path)
    results = model(img, conf=0.5)

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            # Crop plate region
            plate_crop = img[y1:y2, x1:x2]
            if plate_crop.shape[0] < 10 or plate_crop.shape[1] < 20:
                continue

            # Preprocess (paper Eq. 3)
            enhanced = preprocess_plate(plate_crop)

            # EasyOCR (paper Eq. 4)
            ocr_results = reader.readtext(enhanced, detail=0)
            plate_text = ' '.join(ocr_results).upper().strip()
            cleaned = re.sub(r'[^A-Z0-9 ]', '', plate_text)

            # Regex validation (paper Eq. 6)
            is_valid = bool(PLATE_REGEX.match(cleaned))

            print(f"Image: {os.path.basename(img_path)} | Plate: {cleaned} | Conf: {conf:.2f} | Valid: {is_valid}")

            # Annotate image
            color = (0, 255, 0) if is_valid else (0, 0, 255)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, cleaned, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            # Log for CSV (paper Section 7.3)
            csv_rows.append({
                "filename": os.path.basename(img_path),
                "plate_text": cleaned,
                "confidence": round(conf, 4),
                "valid": is_valid,
                "timestamp": datetime.now().isoformat()
            })

    cv2_imshow(img)
    print("---")

# Save CSV output (paper Section 7.3, Fig. 8)
df = pd.DataFrame(csv_rows)
df.to_csv("detections.csv", index=False)
print(f"\nSaved {len(csv_rows)} detections to detections.csv")
df

## 7. Download Trained Weights

Download `best.pt` and place it in your local project:
```
parking-camera-service/models/best.pt
```

In [ ]:
from google.colab import files

# Download the best model weights
files.download("runs/detect/license_plate_detector/weights/best.pt")

# Also download the detection CSV
files.download("detections.csv")

print("Download complete! Place best.pt in: parking-camera-service/models/best.pt")

## 8. (Optional) Save to Google Drive

For persistence across Colab sessions.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Save weights to Google Drive
dest = '/content/drive/MyDrive/parking-anpr-models/'
os.makedirs(dest, exist_ok=True)
shutil.copy('runs/detect/license_plate_detector/weights/best.pt', dest)
shutil.copy('detections.csv', dest)

print(f"Saved to Google Drive: {dest}")